# SheafTSP: Spectral Sheaf Convolution for Cell Complexes

**Track 2 (TNN) submission for the Topological Deep Learning Challenge 2026**

This notebook is an equation-level, executable walkthrough of the SheafTSP
architecture: a spectral sheaf convolutional network with orientation-equivariant
learned transports, a transport-consistency kernel, PPR sheaf diffusion, and an
substructure-counting pathway that is exact under the clique lifting and derived from the complex itself. Every mechanism is demonstrated
on a toy graph, including the property checks (orthogonality, equivariance,
exact counting) that back the claims in `docs/sheaf_tsp_overview.html`.

Official 72-run grid results for this configuration: community detection
accuracy **0.4735**, triangle-count MSE/triangle **0.0108**
(12 GraphUniverse settings x 3 seeds x 2 tasks).

### References

1. Tandon et al. "Consistent Geometric Deep Learning via Hilbert Bundles and Cellular Sheaves" (2026), [arXiv:2605.06395](https://arxiv.org/abs/2605.06395)
2. Bodnar et al. "Neural Sheaf Diffusion" (2022), [arXiv:2202.04579](https://arxiv.org/abs/2202.04579)
3. Bamberger, Barbero, Dong & Bronstein, "Bundle Neural Networks for message diffusion on graphs" (2024), [arXiv:2405.15540](https://arxiv.org/abs/2405.15540)
4. Chen, Chen, Villar & Bruna, "Can Graph Neural Networks Count Substructures?" (NeurIPS 2020), [arXiv:2002.04025](https://arxiv.org/abs/2002.04025)
5. Zhang et al. "MagNet: A Neural Network for Directed Graphs" (NeurIPS 2021), [arXiv:2102.11391](https://arxiv.org/abs/2102.11391)


---
## 1. Architecture Overview

The full pipeline from raw graph to prediction:

```
Raw Graph
   |  CellCliqueLifting: ALL 3-cliques become 2-cells
   |  (canonical set -> the lifted complex is invariant to node relabeling)
   v
Cell Complex  (x_0, x_1, x_2, B_1, B_2, L_1^down, L_1^up)
   |  AllCellFeatureEncoder (width 64, fixed by the challenge)
   v
Encoded features  x_1 in R^{N_1 x 64}
   |  SheafTSP backbone (3 layers):
   |    transports R_e in SO(d)  ->  kernel k_ij on ||s_i - R_e s_j||
   |    ->  L_hat = D^{-1/2} delta^T K delta D^{-1/2}
   |    ->  PPR filter  y = sum_k w_k P^k s,  P = I - L_hat/2,  K = 10
   v
Refined x_1 in R^{N_1 x 64}
   |  x_0 = B_1 @ x_1  +  x_0_enc  +  W_tri t_v      (NO LayerNorm here)
   |          diffusion    residual    exact count signal
   |          t_v = |B_1||B_2|1  (exact under the clique lifting)
   v
Node embeddings  x_0 in R^{N_0 x 64}
   |  readout; graph-level tasks sum-pool (sum preserves count linearity)
   v
Prediction (node-level: community detection / graph-level: triangle count)
```

Two signal regimes coexist as additive terms of one embedding: the
sheaf-diffused signal is operator-normalized for stable long-range diffusion,
and the count term reaches the prediction through strictly linear,
unnormalized operations — any LayerNorm/BatchNorm/mean on that route erases
cardinality (design principles P1-P2 in the overview document).


---
## 2. Mathematical Foundations

### 2.1 Cellular Sheaves (Def. 5 in [1])

A **network sheaf** $\mathcal{F}$ on a graph $G = (V, E)$ assigns:
- A **stalk** $\mathcal{F}(v) \cong \mathbb{R}^d$ to each node $v \in V$
- A **stalk** $\mathcal{F}(e) \cong \mathbb{R}^d$ to each edge $e \in E$
- A **restriction map** $\mathcal{F}_{v \trianglelefteq e} : \mathcal{F}(v) \to \mathcal{F}(e)$ for each incidence $v \trianglelefteq e$

The parameter $d$ is the **stalk dimension** (`stalk_dim` in our code). When $d=1$, the sheaf reduces to a standard graph. When $d \geq 2$, each edge carries a matrix-valued transport that can rotate, reflect, or scale signals.

### 2.2 Coboundary Operator and Sheaf Laplacian (Eq. 8 in [1])

The **coboundary operator** $\delta : C^0(\mathcal{F}) \to C^1(\mathcal{F})$ maps node signals to edge signals:

$$
(\delta \mathbf{x})_e = \mathcal{F}_{u \trianglelefteq e} \, \mathbf{x}_u - \mathcal{F}_{v \trianglelefteq e} \, \mathbf{x}_v \quad \text{for } e = (u, v)
$$

The **Sheaf Laplacian** is:

$$
\mathbf{L}_{\mathcal{F}} = \delta^\top \delta
$$

This is symmetric positive semi-definite by construction ($\mathbf{x}^\top \mathbf{L}_{\mathcal{F}} \mathbf{x} = \|\delta \mathbf{x}\|^2 \geq 0$).

**Block structure.** For an edge $e = (u, v)$ with restriction maps $\mathbf{R}_u \equiv \mathcal{F}_{u \trianglelefteq e}$ and $\mathbf{R}_v \equiv \mathcal{F}_{v \trianglelefteq e}$:

$$
\mathbf{L}_{\mathcal{F}}[u, u] \mathrel{+}= \mathbf{R}_u^\top \mathbf{R}_u, \quad
\mathbf{L}_{\mathcal{F}}[v, v] \mathrel{+}= \mathbf{R}_v^\top \mathbf{R}_v
$$
$$
\mathbf{L}_{\mathcal{F}}[u, v] \mathrel{+}= -\mathbf{R}_u^\top \mathbf{R}_v, \quad
\mathbf{L}_{\mathcal{F}}[v, u] \mathrel{+}= -\mathbf{R}_v^\top \mathbf{R}_u
$$

**Our convention:** We set $\mathbf{R}_v = \mathbf{I}_d$ (identity) and learn only $\mathbf{R}_u$, simplifying the above to:

| Block | Value |
|-------|-------|
| $\mathbf{L}_{\mathcal{F}}[u, u]$ | $+\mathbf{R}^\top \mathbf{R}$ |
| $\mathbf{L}_{\mathcal{F}}[v, v]$ | $+\mathbf{I}_d$ |
| $\mathbf{L}_{\mathcal{F}}[u, v]$ | $-\mathbf{R}^\top$ |
| $\mathbf{L}_{\mathcal{F}}[v, u]$ | $-\mathbf{R}$ |

The resulting $\mathbf{L}_{\mathcal{F}} \in \mathbb{R}^{Nd \times Nd}$ operates on the **stalk-expanded** signal space.

---
## 3. Restriction Map Learning

Each restriction map $\mathbf{R}_e \in SO(d)$ is produced from an
**antisymmetrized** skew-generator and projected to the rotation group.

### Step 1: Antisymmetrized edge conditioning

$$
\mathbf{p}_{uv} = \mathrm{MLP}([\mathbf{x}_u \| \mathbf{x}_v]) - \mathrm{MLP}([\mathbf{x}_v \| \mathbf{x}_u])
$$

so $\mathbf{p}_{vu} = -\mathbf{p}_{uv}$ by construction. The parameters fill a
skew-symmetric matrix $\mathbf{S} = -\mathbf{S}^\top$.

### Step 2: Projection to SO(d)

Default (Cayley): $\mathbf{R} = (\mathbf{I} - \mathbf{S})(\mathbf{I} + \mathbf{S})^{-1}$.
Alternative (`rotation_param: exp`): $\mathbf{R} = \exp(\mathbf{S})$, surjective onto $SO(d)$.

### Properties (all verified in code below)

- **Special orthogonal**: $\mathbf{R}^\top\mathbf{R} = \mathbf{I}$, $\det \mathbf{R} = +1$.
- **Orientation equivariance**: the antisymmetric generator gives
  $\mathbf{R}_{vu} = \mathbf{R}_{uv}^{-1}$ analytically — reversing an edge inverts its
  transport, and the model is invariant to node relabeling.
- **Precision note**: the Cayley image is the dense subset of $SO(d)$ excluding
  rotations with a $-1$ eigenvalue. We benchmarked the surjective exponential
  map head-to-head; the exclusion is measurably non-binding (details in the
  overview document, Sec. 3.1) and Cayley remains the default.
- At $d = 2$, $SO(2)$ is abelian and the resulting operator is real-conjugate
  to a **magnetic Laplacian** with learned per-edge phases [5]. Measured on
  trained models: low-homophily settings learn near-antipodal transports
  (mean $|\theta| = 165.8°$), homophilous settings learn near-identity ones
  ($9.9°$) — the model selects its gauge from data.


---
## 4. Polynomial Spectral Filter (Eq. 10 in [1])

Each `SheafConvLayer` applies a learnable polynomial filter on the Sheaf Laplacian:

$$
\mathbf{y} = \sum_{k=0}^{K-1} c_k \, \mathbf{L}_{\mathcal{F}}^k \, (\mathbf{x} \, \mathbf{W})
$$

where:
- $\mathbf{x} \in \mathbb{R}^{N \times C_{\text{in}}}$ are input features
- $\mathbf{W} \in \mathbb{R}^{C_{\text{in}} \times C_{\text{out}}}$ is a learnable linear transform
- $c_k \in \mathbb{R}$ are learnable filter coefficients
- $K$ is the filter order (`filter_order`, default 3)

**Initialization:** $c_0 = 1, \; c_{k>0} = 0$ (identity filter at init).

### Stalk lifting and pooling

Since $\mathbf{L}_{\mathcal{F}} \in \mathbb{R}^{Nd \times Nd}$ operates on the stalk-expanded space, we:

1. **Lift** features by replicating each node's features across its $d$ stalk dimensions:
   $$\tilde{\mathbf{x}} = \text{repeat\_interleave}(\mathbf{x}\mathbf{W}, d) \in \mathbb{R}^{Nd \times C_{\text{out}}}$$

2. **Apply** the polynomial filter in stalk space:
   $$\tilde{\mathbf{y}} = \sum_k c_k \, \mathbf{L}_{\mathcal{F}}^k \, \tilde{\mathbf{x}} \in \mathbb{R}^{Nd \times C_{\text{out}}}$$

3. **Pool** back to node space by averaging over stalk dimensions:
   $$\mathbf{y}_v = \frac{1}{d} \sum_{i=1}^{d} \tilde{\mathbf{y}}_{v \cdot d + i} \in \mathbb{R}^{C_{\text{out}}}$$

### Computational cost

- Each power $\mathbf{L}_{\mathcal{F}}^k \tilde{\mathbf{x}}$ is computed iteratively: $\mathbf{L}_{\mathcal{F}}^k \tilde{\mathbf{x}} = \mathbf{L}_{\mathcal{F}} (\mathbf{L}_{\mathcal{F}}^{k-1} \tilde{\mathbf{x}})$
- For sparse $\mathbf{L}_{\mathcal{F}}$: $O(K \cdot \text{nnz}(\mathbf{L}_{\mathcal{F}}) \cdot C_{\text{out}})$
- Dense/sparse switch at threshold $Nd \leq 2000$

### Connection to ChebNet

This is a sheaf generalization of ChebNet (Defferrard et al., 2016). When $d = 1$ and all restriction
maps are $\mathbf{R} = 1$, the Sheaf Laplacian reduces to the standard graph Laplacian and the filter
becomes a standard Chebyshev polynomial filter.

---
## 5. Full Layer Computation

Each `SheafConvLayer` performs the following sequence:

$$
\boxed{
\begin{aligned}
\mathbf{R}_e &= \text{Cayley}\big(\text{MLP}([\mathbf{x}_u \| \mathbf{x}_v])\big)
  & \text{(restriction maps)} \\
\mathbf{L}_{\mathcal{F}} &= \delta^\top \delta \quad \text{from } \{\mathbf{R}_e\}_{e \in E}
  & \text{(Sheaf Laplacian)} \\
\mathbf{x}' &= \text{Dropout}(\mathbf{x}) \cdot \mathbf{W}
  & \text{(linear transform)} \\
\tilde{\mathbf{x}} &= \text{lift}(\mathbf{x}') \in \mathbb{R}^{Nd \times C}
  & \text{(stalk expansion)} \\
\tilde{\mathbf{y}} &= \sum_{k=0}^{K-1} c_k \, \mathbf{L}_{\mathcal{F}}^k \, \tilde{\mathbf{x}}
  & \text{(spectral filter)} \\
\mathbf{y} &= \text{pool}(\tilde{\mathbf{y}}) + \mathbf{b}
  & \text{(stalk pooling + bias)} \\
\mathbf{y} &= \text{LayerNorm}(\mathbf{y})
  & \text{(normalization)}
\end{aligned}
}
$$

The full `SheafTSP` backbone stacks $L$ such layers with **residual connections**:

$$
\mathbf{x}^{(l+1)} = \text{ReLU}\big(\mathbf{x}^{(l)} + \text{SheafConvLayer}^{(l)}(\mathbf{x}^{(l)})\big)
$$

(ReLU is omitted after the final layer unless `last_act=True`.)

### Submission defaults on top of the polynomial form

**Transport-consistency kernel.** Edge weights use the distance *under the
learned map*, $\rho_{ij}^2 = \lVert s_i - R_e s_j \rVert^2$, in a Gaussian
kernel with learnable bandwidth — a raw feature-distance kernel would
reintroduce the homophily bias the sheaf exists to remove.

**PPR scalar filter (default, `filter_basis: ppr`, K = 10).** Instead of
per-order weight matrices, the shipped filter uses K+1 scalars on the sheaf
lazy walk $P = I - \tfrac12\hat L_{\mathcal F}$ (spectrum in $[0,1]$, so ten
hops stay numerically stable), initialized to the personalized-PageRank
profile $w_k = \alpha(1-\alpha)^k$ — long-range low-pass diffusion
concentrating on the bottom eigensections, where transport-consistent
community structure lives.


---
## 6. Why This Avoids Oversmoothing (Theorem 1 in [1])

Standard GNNs smooth features by averaging neighbors. After $k$ layers, node features
converge to the null space of the graph Laplacian (a constant vector for connected graphs),
destroying all discriminative signal.

The **Sheaf Laplacian** has a richer null space. A signal $\mathbf{x}$ satisfies
$\mathbf{L}_{\mathcal{F}} \mathbf{x} = 0$ if and only if:

$$
\mathbf{R}_u \mathbf{x}_u = \mathbf{x}_v \quad \forall \, e = (u,v) \in E
$$

These are called **global sections** of the sheaf. Unlike the graph Laplacian's null space
(constant signals), the sheaf's null space is controlled by the restriction maps.
If the maps are non-trivial (rotations, reflections), global sections can encode
rich, non-constant features.

**Effective spectral rank:** With stalk dimension $d$, the Sheaf Laplacian has approximately
$d \times$ as many distinct eigenvalues as the standard Laplacian, giving the polynomial filter
$d \times$ as many frequency components to work with.

**Convergence guarantee (Theorem 1):** As the graph discretization refines, the Sheaf Laplacian
converges to the **Connection Laplacian** on the underlying manifold, providing a principled
geometric foundation for the learned transport maps.

---
## 7. Implementation Walkthrough

Below we import the components and walk through each computation with a small example.

In [ ]:
import sys
from pathlib import Path

# Setup paths
_ROOT = Path.cwd().resolve()
_REPO = _ROOT if (_ROOT / "configs" / "run.yaml").exists() else _ROOT.parent
if str(_REPO) not in sys.path:
    sys.path.insert(0, str(_REPO))

import torch
import torch.nn as nn
import torch.nn.functional as F

from topobench.nn.backbones.cell.sheaf_tsp import (
    RestrictionMapLearner,
    SheafConvLayer,
    SheafTSP,
    build_sheaf_laplacian_torch,
    build_sheaf_laplacian_sparse,
)

torch.manual_seed(42)
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")

### 7.1 Create a toy graph

We create a small 5-node graph to trace every computation explicitly.

```
  0 --- 1
  |     |
  2 --- 3
    \   |
      4
```

6 directed edges (each undirected edge appears once for simplicity).
Feature dimension $C = 8$, stalk dimension $d = 2$.

In [ ]:
N = 5          # nodes
C = 8          # feature channels
d = 2          # stalk dimension
K = 3          # filter order

# Node features (random for demonstration)
x = torch.randn(N, C)
print(f"Node features x: {x.shape}")

# Edge index (directed): (0,1), (0,2), (1,3), (2,3), (2,4), (3,4)
edge_index = torch.tensor([
    [0, 0, 1, 2, 2, 3],
    [1, 2, 3, 3, 4, 4],
], dtype=torch.long)
E = edge_index.shape[1]
print(f"Edges: {E}")
print(f"Stalk-expanded space: N*d = {N*d}")

### 7.2 Step 1: Learn Restriction Maps

The `RestrictionMapLearner` produces one $d \times d$ orthogonal matrix per edge:

$$
\mathbf{R}_e = \text{Cayley}\big(\text{MLP}([\mathbf{x}_{\text{src}(e)} \| \mathbf{x}_{\text{dst}(e)}])\big) \in O(d)
$$

In [ ]:
map_learner = RestrictionMapLearner(in_channels=C, stalk_dim=d)
R = map_learner(x, edge_index)
print(f"Restriction maps R: {R.shape}  (E={E}, d={d}, d={d})")

# Verify orthogonality: R^T R should be close to I
for e_idx in range(E):
    RtR = R[e_idx].T @ R[e_idx]
    err = (RtR - torch.eye(d)).abs().max().item()
    print(f"  Edge {e_idx}: ||R^T R - I||_max = {err:.2e}")

In [ ]:
# Property checks: special orthogonality and orientation equivariance
R_fwd = map_learner(x, edge_index)                 # R_uv
R_bwd = map_learner(x, edge_index.flip(0))         # R_vu

I = torch.eye(d).expand(edge_index.shape[1], d, d)
orth_err = (R_fwd.transpose(1, 2) @ R_fwd - I).abs().max()
dets = torch.linalg.det(R_fwd)
equiv_err = (torch.bmm(R_fwd, R_bwd) - I).abs().max()

print(f"orthogonality  max|R^T R - I| = {orth_err:.2e}")
print(f"determinants   {dets.tolist()}")
print(f"equivariance   max|R_uv R_vu - I| = {equiv_err:.2e}   (R_vu = R_uv^-1)")


### 7.3 Step 2: Build the Sheaf Laplacian

Given restriction maps $\{\mathbf{R}_e\}$, we construct $\mathbf{L}_{\mathcal{F}} \in \mathbb{R}^{Nd \times Nd}$.

For each edge $e = (u, v)$ with map $\mathbf{R}$:

| Block position | Contribution |
|---|---|
| $(u, u)$ diagonal | $+ \mathbf{R}^\top \mathbf{R}$ |
| $(v, v)$ diagonal | $+ \mathbf{I}_d$ |
| $(u, v)$ off-diagonal | $- \mathbf{R}^\top$ |
| $(v, u)$ off-diagonal | $- \mathbf{R}$ |

**Key properties to verify:**
- Symmetry: $\mathbf{L}_{\mathcal{F}} = \mathbf{L}_{\mathcal{F}}^\top$
- Positive semi-definite: all eigenvalues $\geq 0$

In [ ]:
L_dense = build_sheaf_laplacian_torch(N, edge_index, R, d)
print(f"Sheaf Laplacian L_F: {L_dense.shape}")

# Verify symmetry
sym_err = (L_dense - L_dense.T).abs().max().item()
print(f"Symmetry error: {sym_err:.2e}")

# Verify PSD (all eigenvalues >= 0)
eigvals = torch.linalg.eigvalsh(L_dense)
print(f"Eigenvalue range: [{eigvals.min().item():.6f}, {eigvals.max().item():.6f}]")
print(f"Number of zero eigenvalues (< 1e-6): {(eigvals < 1e-6).sum().item()}")
print(f"\nThese zero eigenvalues correspond to global sections of the sheaf.")

# Compare sparse and dense
L_sparse = build_sheaf_laplacian_sparse(N, edge_index, R, d)
diff = (L_sparse.to_dense() - L_dense).abs().max().item()
print(f"\nSparse vs Dense max difference: {diff:.2e}")

### 7.4 Step 3: Linear Transform and Stalk Lifting

Before filtering, features are linearly projected and "lifted" to the stalk-expanded space:

$$
\mathbf{x}' = \mathbf{x} \cdot \mathbf{W} \in \mathbb{R}^{N \times C_{\text{out}}}
$$

$$
\tilde{\mathbf{x}} = \text{repeat\_interleave}(\mathbf{x}', d) \in \mathbb{R}^{Nd \times C_{\text{out}}}
$$

This replicates node $v$'s features into positions $[vd, vd+1, \ldots, vd+(d-1)]$, so that the
Sheaf Laplacian's block structure can operate on them.

In [ ]:
W = nn.Linear(C, C, bias=False)
x_proj = W(x)  # (N, C)
print(f"After linear: {x_proj.shape}")

x_stalk = x_proj.repeat_interleave(d, dim=0)  # (N*d, C)
print(f"After stalk lift: {x_stalk.shape}")

# Show that node 0's features are replicated across stalk dims 0 and 1
print(f"\nNode 0 features: {x_proj[0, :3].tolist()}")
print(f"Stalk[0] (= node 0, dim 0): {x_stalk[0, :3].tolist()}")
print(f"Stalk[1] (= node 0, dim 1): {x_stalk[1, :3].tolist()}")
print("(Identical -- the Sheaf Laplacian will differentiate them via R)")

### 7.5 Step 4: Polynomial Spectral Filter

The core computation. We iteratively compute powers of $\mathbf{L}_{\mathcal{F}}$ applied to the signal:

$$
\mathbf{y} = c_0 \tilde{\mathbf{x}} + c_1 \mathbf{L}_{\mathcal{F}} \tilde{\mathbf{x}} + c_2 \mathbf{L}_{\mathcal{F}}^2 \tilde{\mathbf{x}} + \cdots
$$

At initialization ($c_0 = 1, c_{k>0} = 0$), the filter is the identity. During training,
the network learns to combine different "frequencies" of the sheaf spectrum:

- **$c_0$ (constant):** Preserves the input signal
- **$c_1$ (linear):** 1-hop sheaf diffusion (neighbors, weighted by $\mathbf{R}$)
- **$c_2$ (quadratic):** 2-hop sheaf diffusion
- Higher $k$: larger receptive field

In [ ]:
# Simulate the polynomial filter
filter_coeffs = torch.zeros(K)
filter_coeffs[0] = 1.0  # identity init

# Iterative power application
y = filter_coeffs[0] * x_stalk  # c_0 * L^0 * x = c_0 * x
Lx = x_stalk
for k in range(1, K):
    Lx = L_dense @ Lx  # L^k * x = L * (L^{k-1} * x)
    y = y + filter_coeffs[k] * Lx
    print(f"  k={k}: ||L^{k} x||_F = {Lx.norm():.4f}, c_{k} = {filter_coeffs[k]:.4f}")

print(f"\nFiltered signal (stalk space): {y.shape}")

# Pool back to node space: average over stalk dims
y_node = y.view(N, d, -1).mean(dim=1)
print(f"After stalk pooling: {y_node.shape}")

### 7.6 Full Forward Pass

Now we run the complete `SheafConvLayer` and then the full `SheafTSP` backbone with residual connections.

In [ ]:
# Single layer
layer = SheafConvLayer(in_channels=C, out_channels=C, stalk_dim=d, filter_order=K)
y_layer = layer(x, edge_index)
print(f"Single layer output: {y_layer.shape}")

# Full backbone with residual connections
n_layers = 3
backbone = SheafTSP(in_channels=C, n_layers=n_layers, stalk_dim=d, filter_order=K)

# Create dummy Hodge Laplacians (SheafTSP extracts edge_index from L1 = Ld + Lu)
# In practice, TopoBench provides these from the cell complex lifting
Ld = torch.sparse_coo_tensor(
    edge_index, torch.ones(E), (N, N)
).coalesce()
Lu = torch.sparse_coo_tensor(
    torch.zeros(2, 0, dtype=torch.long), torch.zeros(0), (N, N)
).coalesce()

y_full = backbone(x, Ld, Lu)
print(f"Full backbone output: {y_full.shape}")
print(f"Output dimension matches input: {y_full.shape == x.shape}")

### 7.7 Parameter Count

SheafTSP is deliberately lightweight. The parameter budget per layer:

| Component | Formula | Default ($C=32, d=2, K=3$) |
|---|---|---|
| Restriction MLP | $(2C)(4d) + 4d \cdot \frac{d(d-1)}{2}$ | 264 |
| Linear $\mathbf{W}$ | $C \times C$ | 1,024 |
| Filter coefficients $c_k$ | $K$ | 3 |
| LayerNorm | $2C$ | 64 |
| Bias | $C$ | 32 |
| **Per layer** | | **~1,387** |
| **3 layers total** | | **~4,161** |

The feature encoder adds separate parameters for projecting raw features into the 32-dim space.

In [ ]:
total = sum(p.numel() for p in backbone.parameters())
trainable = sum(p.numel() for p in backbone.parameters() if p.requires_grad)
print(f"Total parameters: {total:,}")
print(f"Trainable parameters: {trainable:,}")

print("\nPer-layer breakdown:")
for i, layer in enumerate(backbone.layers):
    n_params = sum(p.numel() for p in layer.parameters())
    print(f"  Layer {i}: {n_params:,} params")
    for name, p in layer.named_parameters():
        print(f"    {name}: {p.shape} ({p.numel()} params)")

---
## 7.8 Counting substructures exactly

Message-passing GNNs provably cannot count triangles [4]; lifted models
receive higher-order structure explicitly. SheafTSP derives a per-node count
signal from the lifted complex's own incidence matrices:

$$
t_v = |B_1|\,|B_2|\,\mathbf 1, \qquad \sum_{v} t_v = 6 \cdot \#\text{triangles}
$$

Under the all-3-cliques lifting each triangle incident to a node contributes
exactly two of its edges at that node, so the identity is exact — verified
below against direct enumeration. The signal is injected as
$x_0 \mathrel{+}= W_{\text{tri}} t_v$ (zero-initialized, one warm channel)
and travels a strictly linear, unnormalized route to the sum-pooled
prediction: the regression head only learns a scale factor.


In [ ]:
import networkx as nx
from torch_geometric.data import Data
from topobench.transforms.liftings.graph2cell.clique_cell_lifting import CellCliqueLifting

g = nx.gnp_random_graph(30, 0.25, seed=7)
true_triangles = sum(nx.triangles(g).values()) // 3

E = torch.tensor(list(g.edges())).t()
E = torch.cat([E, E.flip(0)], dim=1)
lifted = CellCliqueLifting()(Data(edge_index=E, x=torch.randn(30, 4), num_nodes=30))

B1 = lifted.incidence_1.to_dense().abs()
B2 = lifted.incidence_2.to_dense().abs()
t_v = B1 @ B2 @ torch.ones(B2.shape[1], 1)

print(f"2-cells attached by the clique lifting : {B2.shape[1]}")
print(f"true triangle count (networkx)         : {true_triangles}")
print(f"sum(t_v) / 6                           : {t_v.sum().item() / 6:.0f}   (exact)")


---
## 8. Integration into TopoBench

### 8.1 SheafTSPWrapper

The wrapper bridges TopoBench's batch format to the backbone:

```python
x_1 = backbone(batch.x_1, batch.down_laplacian_1, batch.up_laplacian_1)
x_0 = B_1 @ x_1   # propagate to nodes via boundary operator
```

The boundary operator $\mathbf{B}_1 \in \mathbb{R}^{N_0 \times N_1}$ maps 1-cell (edge) embeddings
to 0-cell (node) embeddings. Each node receives the sum of its incident edge features.

### 8.2 Readout: PropagateSignalDown

After obtaining $\mathbf{x}_0$ (node embeddings), the `PropagateSignalDown` readout:
1. Pools node embeddings per graph: $\mathbf{h}_G = \text{sum}_{v \in G}(\mathbf{x}_0^v)$
2. Passes through an MLP classifier for the final prediction

### 8.3 Loss and Training

TopoBench uses its standard `TBLoss`:
- **Classification:** Cross-entropy on graph-level labels
- **Optimizer:** Adam with default TopoBench learning rate schedule
- **Early stopping** on validation loss (patience from config)

---
## 9. From Raw Graph to Cell Complex

TopoBench transforms raw graphs into cell complexes via **cycle lifting**:

1. **Input graph** $G = (V, E)$ with node features and labels
2. **Cycle lifting** (`graph2cell/cycle_lifting`) detects cycles and promotes them to 2-cells
3. **Cell complex** $(X, \mathcal{X})$ with:
   - 0-cells (nodes): $x_0 \in \mathbb{R}^{N_0 \times F_0}$
   - 1-cells (edges): $x_1 \in \mathbb{R}^{N_1 \times F_1}$ (initialized from node features)
   - Incidence matrix $\mathbf{B}_1 \in \mathbb{R}^{N_0 \times N_1}$ (boundary operator)
   - Down Laplacian $\mathbf{L}_1^{\text{down}} = \mathbf{B}_1^\top \mathbf{B}_1$
   - Up Laplacian $\mathbf{L}_1^{\text{up}} = \mathbf{B}_2 \mathbf{B}_2^\top$ (if 2-cells exist)

4. **Feature encoding** (`AllCellFeatureEncoder`) projects raw features to $C = 32$ dimensions

### What SheafTSP uses from the cell complex

| Attribute | Shape | Purpose |
|---|---|---|
| `x_1` | $(N_1, C)$ | 1-cell features (backbone input) |
| `down_laplacian_1` | $(N_1, N_1)$ sparse | Combined with `up_laplacian_1` to get edge adjacency |
| `up_laplacian_1` | $(N_1, N_1)$ sparse | Combined with `down_laplacian_1` to get edge adjacency |
| `incidence_1` | $(N_0, N_1)$ sparse | Boundary map for node embedding recovery |
| `y` | $(N_G,)$ | Ground truth labels |
| `batch_0` | $(N_0,)$ | Batch assignment for pooling |

The backbone extracts `edge_index` from the nonzero pattern of $\mathbf{L}_1 = \mathbf{L}_1^{\text{down}} + \mathbf{L}_1^{\text{up}}$,
giving the adjacency structure of the 1-skeleton. This is then used to learn restriction maps
between adjacent 1-cells.

---
## 10. GraphUniverse Evaluation Grid

The challenge evaluation trains and tests across synthetic graph distributions:

| Parameter | Levels | Values |
|---|---|---|
| Homophily | 3 | low [0.0, 0.1], mid [0.4, 0.6], high [0.9, 1.0] |
| Avg degree | 2 | low [1, 2], high [4, 5] |
| Power-law exponent | 2 | low [1.5, 2.0], high [4.0, 5.0] |

**Grid:** 3 homophily x 2 degree x 2 power-law = 12 settings

**Tasks:** community detection, node classification (2 modes)

**Seeds:** 3 random seeds per configuration

**Total:** 12 x 2 x 3 = **72 training runs**

Each trained model is then evaluated on all 12 settings (OOD cross-evaluation),
producing a 12x12 performance matrix per task.

---
## 11. Model Configuration

The submitted Hydra defaults (`configs/model/cell/sheaf_tsp.yaml`):

```yaml
backbone:
  n_layers: 3            # sheaf convolution layers
  stalk_dim: 2           # SO(2) transports (magnetic-Laplacian regime)
  filter_basis: ppr      # scalar PPR coefficients on P = I - L_hat/2
  filter_order: 10       # ten stable diffusion hops
  kernel_distance: transport   # ||s_i - R_e s_j|| in the Gaussian kernel
  reg_form: alignment    # bounded kernel-alignment transport regularizer
  rotation_param: cayley # benchmarked against the surjective exp map
  dropout: 0.0           # signal-path dropout corrupts the count pathway
  mlp_dropout: 0.5       # regularize the transport MLP instead

backbone_wrapper:
  residual_connections: false  # no post-hoc LayerNorm on x_0 (P2)
  count_source: incidence      # t_v = |B1||B2|1, endogenous and exact
  tri_warm: 0.1                # warm-start channel of W_tri

transforms (model_defaults):
  graph2cell_lifting: clique_cell   # all 3-cliques; permutation-invariant
```

Every default was selected against a measured alternative; the experiment
log records each comparison, including the rejected options
(`docs/BATTLE_PLAN.md` in the project repository).


---
## 12. Run Evaluation

The cells below run the standard challenge evaluation grid. Set `MODEL_CONFIG` and execute.

In [ ]:
import os
os.environ["WANDB_MODE"] = "disabled"
os.environ["WANDB_SILENT"] = "true"

from utils import (
    resolve_project_root,
    run_challenge_grid,
    save_challenge_artifacts,
)

PROJECT_ROOT = resolve_project_root(_REPO)
MODEL_CONFIG = "cell/sheaf_tsp"

In [ ]:
results, study_id = run_challenge_grid(
    project_root=PROJECT_ROOT,
    model_config=MODEL_CONFIG,
    extra_overrides=["trainer=cpu"],  # change to "trainer=gpu" for GPU
    quiet=True,
)

In [ ]:
output_paths = save_challenge_artifacts(
    results,
    model_config=MODEL_CONFIG,
    study_id=study_id,
)

print(f"Results saved to: {output_paths['dir']}")
print(f"JSON: {output_paths['json']}")

In [ ]:
import pandas as pd

df = pd.DataFrame(results)
print(f"Completed {len(df)} runs")
df.head(10)

---
## 13. Summary

**SheafTSP** couples two signal regimes in one architecture:

1. **Learned sheaf diffusion** — orientation-equivariant $SO(2)$ transports
   (antisymmetrized Cayley generator, $R_{vu} = R_{uv}^{-1}$ by construction),
   a transport-consistency kernel, and PPR diffusion on the sheaf lazy walk.
   Measured transports match the theory: near-antipodal on heterophilic
   settings, near-identity on homophilous ones.

2. **An exact counting pathway** — $t_v = |B_1||B_2|\mathbf 1$ from the
   canonical clique lifting, exact by construction and preserved by a
   strictly linear, unnormalized route to the prediction.

The full pipeline is invariant to node relabeling (equivariant transports +
canonical lifting), its operator spectrum is bounded in $[0,2]$ per input
(kernel-weighted degrees, verified numerically), and the official 72-run grid
scores are community detection **0.4735** and triangle-count MSE/triangle
**0.0108**. Design principles, scoped theoretical claims, and the complete
experiment log live in the project documentation.
